<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Score the range

**[Score the range](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/)** · Forecasting and Time Series: Predict What Comes Next · Module 7, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score a forecast range with its coverage and the pinball loss, compare the ranges of two methods, and test conformal intervals, a simple fix for a range that is too narrow.

| | |
|---|---|
| **Time** | About 35 minutes, of which about 2 minutes of waiting for the backtests |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Ranges, quantiles and coverage, from [A forecast is a range](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M07-L02-score-the-range/score-the-range-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The pinball loss on one day

A **quantile forecast** at the level 0.9 (q90) says: "the real value is below this number 9 days out of 10". The **pinball loss** scores it. If the real value is above the forecast, each unit of the gap costs 0.9. If it is below, each unit costs only 0.1. So a q90 that is too low is punished 9 times more than one that is too high.

`bt.pinball(y, q, level)` computes it. The q90 forecast is 26.

> **Predict.** The real value is 20 on one day and 30 on another. Which day has the larger loss, and how many times larger? Then run the cell.

In [ ]:
for real in [20, 30]:
    print(f"real {real}, q90 26: loss {float(bt.pinball(real, 26, 0.9)):.1f}")

> **Check.** You should see `loss 0.6` for 20 (6 units too high × 0.1) and `loss 3.6` for 30 (4 units too low × 0.9).

The day of 30 costs 6 times more, with a gap that is smaller (4 units instead of 6). The loss is low only when the q90 sits where the real value is below it about 9 days in 10. For the median (q50), both sides cost 0.5: the pinball loss is then half the absolute error.

## 2. Two methods, two ranges

The next cell runs the course backtest (13 cut-offs, 50 products, 14 days) twice: the **seasonal naive** forecast (the same weekday last week) and **ETS**, both with their 80% and 90% ranges from statsforecast. Then it prints the coverage of each range and the pinball loss of the five quantiles (`bt.score` divides it by the mean real value, so products of every size count the same way).

> **Predict.** Which method has the wider ranges? Which one has the lower pinball loss? Then run the cell. It takes about a minute.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive

def ranged(make_model, name):
    """A forecaster for the backtest: the model's forecast and its 80% and 90% ranges as quantile columns."""
    def forecaster(train, h, future):
        f = StatsForecast(models=[make_model()], freq="D").forecast(
            df=train[["unique_id", "ds", "y"]], h=h, level=[80, 90])
        out = pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[name],
                            "q5": f[f"{name}-lo-90"], "q10": f[f"{name}-lo-80"], "q50": f[name],
                            "q90": f[f"{name}-hi-80"], "q95": f[f"{name}-hi-90"]})
        return out.assign(**{c: out[c].clip(lower=0) for c in ["yhat", "q5", "q10", "q50", "q90", "q95"]})
    return forecaster

sales = load_sales(BACKTEST_PRODUCTS)
fc_snaive = bt.backtest(sales, ranged(lambda: SeasonalNaive(season_length=7), "SeasonalNaive"), h=14, origins=ORIGINS)
fc_ets = bt.backtest(sales, ranged(lambda: AutoETS(season_length=7), "AutoETS"), h=14, origins=ORIGINS)
rows = []
for name, fc in [("seasonal naive", fc_snaive), ("ETS", fc_ets)]:
    width = (fc["q90"] - fc["q10"]).mean()
    rows.append({"method": name, "coverage 80%": bt.coverage(fc, "q10", "q90"), "coverage 90%": bt.coverage(fc, "q5", "q95"),
                 "80% range width": round(float(width), 1), "pinball": bt.score(fc, sales, name, season=7)["pinball"]})
print(pd.DataFrame(rows).to_string(index=False))

> **Check.** You should see this table:
>
> ```text
>         method  coverage 80%  coverage 90%  80% range width  pinball
> seasonal naive         0.877         0.929             13.5    0.133
>            ETS         0.835         0.905              9.6    0.110
> ```

Both 80% ranges hold a little **more** than 80% of the real values: they are a little too wide. The seasonal naive range is wider (13.5 units against 9.6), and it holds 87.7% instead of 80%. Its pinball loss is higher too: a range that is too wide also costs points, because its q10 and q90 sit far from where the real values fall. A good range is **calibrated** (the coverage is close to the promise) and **sharp** (as narrow as it can be while it keeps its promise). The pinball loss rewards both.

## 3. Where do the misses fall?

Coverage counts the real values inside the range. It does not say whether the misses are above or below. For a stock order, misses above the range (demand higher than the forecast) are the ones that cause stock-outs.

> **Your turn.** For the ETS forecasts `fc_ets`, compute the share of real values **above** the q90 column. Put it in `above_q90`, rounded to 3 digits. Then run the check cell.

In [ ]:
above_q90 = round(float((fc_ets["y"] > fc_ets["q90"]).mean()), 3)
print("above q90:", above_q90, " below q10:", round(float((fc_ets["y"] < fc_ets["q10"]).mean()), 3))

In [ ]:
expect('above_q90', above_q90, 0.088)

A q90 should have 10% of the real values above it. ETS has 8.8% above its q90 and 7.7% below its q10. Both sides are a little under 10%, so the range is a little wide on both sides, and not skewed to one side.

## 4. Failure case: conformal intervals with too few windows

**Conformal intervals** make a range from the model's own past errors. statsforecast runs the model on a few earlier windows of history (here 4 windows of 14 days), measures its errors there, and puts the range around the new forecast from the size of those errors. It needs no assumption about the shape of the errors, so it is a common fix for a range that is too narrow.

The next cell runs ETS with conformal intervals on the last 4 cut-offs of the backtest, and compares it with ETS's own range on the same 4 cut-offs. Conformal intervals fit the model 5 times at each cut-off, so we use 4 cut-offs to keep the wait short.

> **Predict.** Will the conformal range hold 80% of the real values? Then run the cell. It takes about a minute and a half.

In [ ]:
from statsforecast.models import ConformalIntervals

last4 = ORIGINS[-4:]
fc_conf = bt.backtest(sales, ranged(lambda: AutoETS(season_length=7, prediction_intervals=ConformalIntervals(h=14, n_windows=4)),
                                    "AutoETS"), h=14, origins=last4)
own = fc_ets[fc_ets["origin"].isin(last4)]
for name, fc in [("ETS, own range", own), ("ETS, conformal (4 windows)", fc_conf)]:
    print(f"{name:28} coverage 80%: {bt.coverage(fc, 'q10', 'q90')}  90%: {bt.coverage(fc, 'q5', 'q95')}  "
          f"pinball: {bt.score(fc, sales, name, season=7)['pinball']}")

> **Check.** You should see `coverage 80%: 0.843  90%: 0.909` for ETS's own range and `coverage 80%: 0.684  90%: 0.75` for the conformal range.

The "fix" made the range worse: the conformal 90% range holds only 75% of the real values. The reason is the number of windows. For each day ahead, the method has only 4 past errors. The largest of 4 errors is passed by a new error about 1 time in 5. So a range built from 4 errors cannot hold 90%, whatever the model. Conformal intervals need many windows (many past errors) to keep their promise, and each window is one more fit of the model.

## Next

You can now score a range with coverage and the pinball loss, and you saw a range "fix" fail its own test. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/) has the diagrams and the knowledge checks that count toward your certificate. The next lesson turns the range into Kenji's order and Grace's roster.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [From forecast to order and roster](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.